In [7]:
import copy
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [8]:
dataset = pd.read_csv("dataset_meta.csv", dtype={"Gene ID": str})

X_ntv3 = np.load("ntv3_embeddings.npy").astype(np.float32)
X_esm2 = np.load("esm2_embeddings.npy").astype(np.float32)

if X_ntv3.shape[0] != X_esm2.shape[0]:
    raise ValueError(
        f"Embedding row mismatch: NTv3 {X_ntv3.shape[0]} vs ESM2 {X_esm2.shape[0]}"
    )

if len(dataset) != X_ntv3.shape[0]:
    raise ValueError(
        f"dataset_meta.csv has {len(dataset)} rows, embeddings have {X_ntv3.shape[0]}"
    )

y = dataset["label"].to_numpy()

print("Loaded ntv3_embeddings.npy:", X_ntv3.shape)
print("Loaded esm2_embeddings.npy:", X_esm2.shape)
print("Genes:", len(dataset))
print(dataset["label"].value_counts())

Loaded ntv3_embeddings.npy: (18015, 768)
Loaded esm2_embeddings.npy: (18015, 1280)
Genes: 18015
label
0    16506
1     1509
Name: count, dtype: int64


In [9]:
PPI_PATH = "BIOGRID-ALL-3.5.181.tab2.txt"
HUMAN_TAXON = "9606"

ppi = pd.read_csv(
    PPI_PATH,
    sep="\t",
    usecols=[
        "Entrez Gene Interactor A",
        "Entrez Gene Interactor B",
        "Experimental System Type",
        "Organism Interactor A",
        "Organism Interactor B",
    ],
    dtype=str,
    low_memory=False,
)

ppi = ppi[
    (ppi["Organism Interactor A"] == HUMAN_TAXON)
    & (ppi["Organism Interactor B"] == HUMAN_TAXON)
    & (ppi["Experimental System Type"].str.lower() == "physical")
].copy()

ppi["Entrez Gene Interactor A"] = ppi["Entrez Gene Interactor A"].str.strip()
ppi["Entrez Gene Interactor B"] = ppi["Entrez Gene Interactor B"].str.strip()

gene_ids = dataset["Gene ID"].astype(str).tolist()
id_to_idx = {gid: i for i, gid in enumerate(gene_ids)}
gene_set = set(id_to_idx)

a = ppi["Entrez Gene Interactor A"]
b = ppi["Entrez Gene Interactor B"]
in_dataset = a.isin(gene_set) & b.isin(gene_set) & (a != b)

pairs = pd.DataFrame({"a": a[in_dataset].to_numpy(), "b": b[in_dataset].to_numpy()})
lo = pairs.min(axis=1)
hi = pairs.max(axis=1)
undirected = pd.DataFrame({"a": lo, "b": hi}).drop_duplicates()

neighbors = {gid: set() for gid in gene_set}
for left, right in undirected.itertuples(index=False):
    neighbors[left].add(right)
    neighbors[right].add(left)

X_ntv3_nei = np.zeros_like(X_ntv3, dtype=np.float32)
X_esm2_nei = np.zeros_like(X_esm2, dtype=np.float32)

n_with_partner = 0
for gid, idx in id_to_idx.items():
    partner_idxs = [id_to_idx[p] for p in neighbors[gid]]
    if not partner_idxs:
        continue
    n_with_partner += 1
    X_ntv3_nei[idx] = X_ntv3[partner_idxs].mean(axis=0)
    X_esm2_nei[idx] = X_esm2[partner_idxs].mean(axis=0)

X = np.concatenate(
    [X_ntv3, X_esm2, X_ntv3_nei, X_esm2_nei],
    axis=1,
).astype(np.float32)

print(f"Human physical PPI edges (in dataset): {len(undirected):,}")
print(f"Genes with >= 1 PPI partner: {n_with_partner:,}")
print(f"Isolated genes: {len(dataset) - n_with_partner:,}")
print("X_ntv3 neighbor shape:                 ", X_ntv3_nei.shape)
print("X_esm2 neighbor shape:                 ", X_esm2_nei.shape)
print("X concat shape (own + PPI neighbors):  ", X.shape)
print("y shape:                               ", y.shape)

Human physical PPI edges (in dataset): 312,142
Genes with >= 1 PPI partner: 15,559
Isolated genes: 2,456
X_ntv3 neighbor shape:                  (18015, 768)
X_esm2 neighbor shape:                  (18015, 1280)
X concat shape (own + PPI neighbors):   (18015, 4096)
y shape:                                (18015,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

Training samples: 14412
Testing samples: 3603

Training distribution:
0    13205
1     1207
Name: count, dtype: int64

Testing distribution:
0    3301
1     302
Name: count, dtype: int64


In [11]:
torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)

torch_scaler = StandardScaler()
X_tr_scaled = torch_scaler.fit_transform(X_tr)
X_val_scaled = torch_scaler.transform(X_val)
X_test_scaled_torch = torch_scaler.transform(X_test)

train_ds = TensorDataset(
    torch.tensor(X_tr_scaled, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.long),
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled_torch, dtype=torch.float32, device=device)


class EssentialityMLP(nn.Module):
    def __init__(self, in_features, n_classes=2, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, n_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        return self.fc4(x) 


model = EssentialityMLP(in_features=X_tr.shape[1]).to(device)

print(model)
print("Input features (NTv3 DNA + ESM2 protein + PPI neighbor means):", X_tr.shape[1])
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

n_pos = int((y_tr == 1).sum())
n_neg = int((y_tr == 0).sum())

class_weights = torch.tensor(
    [len(y_tr) / (2 * n_neg), len(y_tr) / (2 * n_pos)],
    dtype=torch.float32,
    device=device,
)

print(f"\nTraining rows: {len(y_tr):,} (essential {n_pos:,} / non-essential {n_neg:,})")
print(f"Validation rows: {len(y_val):,} (essential {int(y_val.sum()):,})")
print(f"Class weights: non-essential {class_weights[0]:.3f} | essential {class_weights[1]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

MAX_EPOCHS = 300
PATIENCE = 30

best_ap = -np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(yb)

    epoch_loss /= len(train_ds)

    model.eval()

    with torch.no_grad():
        val_prob = torch.softmax(model(X_val_tensor), dim=1)[:, 1].cpu().numpy()

    val_ap = average_precision_score(y_val, val_prob)

    if val_ap > best_ap:
        best_ap = val_ap
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:3d} | train loss {epoch_loss:.4f} | val PR-AUC {val_ap:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Validation PR-AUC: {best_ap:.4f} (baseline {y_val.mean():.4f})")

Device: mps
EssentialityMLP(
  (fc1): Linear(in_features=4096, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=1024, bias=True)
  (fc3): Linear(in_features=1024, out_features=512, bias=True)
  (fc4): Linear(in_features=512, out_features=2, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)
Input features (NTv3 DNA + ESM2 protein + PPI neighbor means): 4096
Trainable parameters: 21502466

Training rows: 12,250 (essential 1,026 / non-essential 11,224)
Validation rows: 2,162 (essential 181)
Class weights: non-essential 0.546 | essential 5.970
epoch   1 | train loss 0.5993 | val PR-AUC 0.4759
epoch  10 | train loss 0.4211 | val PR-AUC 0.5343
epoch  20 | train loss 0.2528 | val PR-AUC 0.5071
epoch  30 | train loss 0.1711 | val PR-AUC 0.4863

Early stopping at epoch 39.

Best epoch: 9
Validation PR-AUC: 0.5675 (baseline 0.0837)


In [12]:
model.eval()

with torch.no_grad():
    test_prob = torch.softmax(model(X_test_tensor), dim=1)

    test_pred = test_prob.argmax(dim=1).cpu().numpy()
    test_prob_essential = test_prob[:, 1].cpu().numpy()

print("Classification report (softmax argmax):\n")
print(classification_report(y_test, test_pred, target_names=["Non-essential", "Essential"]))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, test_prob_essential):.4f}")
print(f"PR-AUC baseline (prevalence): {y_test.mean():.4f}")

Classification report (softmax argmax):

               precision    recall  f1-score   support

Non-essential       0.98      0.88      0.93      3301
    Essential       0.37      0.80      0.51       302

     accuracy                           0.87      3603
    macro avg       0.68      0.84      0.72      3603
 weighted avg       0.93      0.87      0.89      3603

Confusion matrix:
[[2894  407]
 [  59  243]]

ROC-AUC: 0.9229
PR-AUC:  0.6225
PR-AUC baseline (prevalence): 0.0838
